In [1]:
# ============================================================
# 02_mlp_cosine.ipynb
# MLP compressor with cosine index — 10k and full 233k
# Includes: no rerank, WJ rerank, sqrt/Bhattacharyya
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import nmslib
import pickle
import time
import os
import psutil
from tqdm import tqdm

device  = torch.device('cuda:0')
THREADS = 32

# ─── Model definition ─────────────────────────────────────────────────────────
class QuadtreeCompressorV1(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False),   nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x, for_index=False):
        out = self.net(x)
        if for_index:
            out = F.relu(out)
            out = out / out.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return out

class QuadtreeCompressorV1Fixed(nn.Module):
    """MLP with log1p input scaling — required for full dataset."""
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False),   nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def forward(self, x, for_index=False):
        x   = torch.log1p(x * 1e6)
        out = self.net(x)
        if for_index:
            out = F.relu(out)
            out = out / out.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return out

# ─── Load from cache ──────────────────────────────────────────────────────────
print("Loading cached data...")
qt_10k     = np.load('/tmp/qt_10k.npy')
qt_full    = np.load('/tmp/qtree_vectors_full.npy')
with open('/tmp/gt_lookup_10k.pkl',  'rb') as f: gt_10k  = pickle.load(f)
with open('/tmp/gt_lookup_full.pkl', 'rb') as f: gt_full = pickle.load(f)

QUERY_START_10K = 8000
QUERY_START_ID  = 187019

corpus_qt_10k  = qt_10k[:QUERY_START_10K]
query_qt_10k   = qt_10k[QUERY_START_10K:]
corpus_qt_full = qt_full[:QUERY_START_ID]
query_qt_full  = qt_full[QUERY_START_ID:]

print(f"10k:  corpus={corpus_qt_10k.shape} | queries={query_qt_10k.shape}")
print(f"Full: corpus={corpus_qt_full.shape} | queries={query_qt_full.shape}")

# ─── Utilities ────────────────────────────────────────────────────────────────
def recall_at_k(gt_lookup, nbrs, query_start_id, K):
    total = 0.0; count = 0
    for i, (ids, _) in enumerate(nbrs):
        qid = query_start_id + i
        gt  = set(gt_lookup.get(qid, [])[:K])
        if not gt: continue
        total += len(gt & set(ids[:K])) / len(gt)
        count += 1
    return total / count if count > 0 else 0.0

def eval_recall_all(gt_lookup, nbrs, query_start_id, max_k=500):
    return {k: recall_at_k(gt_lookup, nbrs, query_start_id, k)
            for k in [10, 50, 100, 500] if k <= max_k}

def get_mem_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

def generate_cosine_embeddings(model, qt_data, device, batch_size=512):
    """Generate L2-normalized cosine embeddings."""
    model.eval()
    all_embs = []
    with torch.no_grad():
        for start in tqdm(range(0, len(qt_data), batch_size), desc="Embedding"):
            batch = torch.tensor(qt_data[start:start+batch_size],
                                 dtype=torch.float32).to(device)
            out   = model(batch)
            emb   = F.normalize(out, dim=1)
            all_embs.append(emb.cpu().numpy())
    return np.vstack(all_embs)

def build_cosine_index(corpus_embs):
    """Build HNSW cosine index."""
    m0  = get_mem_mb()
    idx = nmslib.init(method='hnsw', space='cosinesimil')
    for i in range(len(corpus_embs)):
        idx.addDataPoint(i, corpus_embs[i])
    t0 = time.time()
    idx.createIndex({'M': 20, 'efConstruction': 200, 'post': 1},
                    print_progress=True)
    build_s = time.time() - t0
    idx_mb  = get_mem_mb() - m0
    idx.setQueryTimeParams({'efSearch': 200})
    return idx, build_s, idx_mb

def query_index(idx, query_embs, k=500):
    t0   = time.time()
    nbrs = idx.knnQueryBatch(query_embs, k=k, num_threads=THREADS)
    qps  = len(query_embs) / (time.time() - t0)
    return nbrs, qps

def rerank_wj(query_qt, nbrs_raw, corpus_qt):
    """Per-query WJ reranking — safe, no large allocations."""
    reranked = []
    for i, (ids, _) in enumerate(nbrs_raw):
        q_vec  = query_qt[i]
        c_vecs = corpus_qt[np.array(ids)]
        mins   = np.minimum(q_vec, c_vecs).sum(axis=1)
        maxs   = np.maximum(q_vec, c_vecs).sum(axis=1)
        wj     = mins / np.maximum(maxs, 1e-10)
        order  = np.argsort(-wj)
        reranked.append(([ids[j] for j in order], []))
    return reranked

def print_results(label, rec, qps, build_s, vec_mb, idx_mb, dim):
    print(f"\n  {label}")
    print(f"    Dim={dim} | Vec={vec_mb:.1f}MB | Idx={idx_mb:.1f}MB | "
          f"Build={build_s:.1f}s | QPS={qps:.1f}")
    for k, r in rec.items():
        print(f"    R@{k:<4} = {r:.4f}")

# ─── 10k experiment ───────────────────────────────────────────────────────────
print("\n" + "="*60)
print("MLP + COSINE — 10k dataset")
print("="*60)

# Load 10k model (no log1p needed — 10k vectors have different scale)
model_10k = QuadtreeCompressorV1(in_dim=qt_10k.shape[1], out_dim=512).to(device)
model_10k.load_state_dict(
    torch.load('/tmp/best_compressor_v1_clean.pt', weights_only=True))
model_10k.eval()
print(f"Model loaded: {sum(p.numel() for p in model_10k.parameters()):,} params")

# Generate embeddings
embs_10k    = generate_cosine_embeddings(model_10k, qt_10k, device)
corpus_10k  = embs_10k[:QUERY_START_10K]
query_10k   = embs_10k[QUERY_START_10K:]
vec_mb_10k  = corpus_10k.nbytes / 1024**2
print(f"Embeddings: {embs_10k.shape} | Vec size: {vec_mb_10k:.1f} MB")

# Embedding quality check
gt_sims, rand_sims = [], []
for i in range(200):
    qid    = QUERY_START_10K + i
    pos_id = gt_10k.get(qid, [None])[0]
    if pos_id is None: continue
    rand_id = np.random.randint(0, QUERY_START_10K)
    q = torch.tensor(qt_10k[qid], dtype=torch.float32).unsqueeze(0).to(device)
    p = torch.tensor(qt_10k[pos_id], dtype=torch.float32).unsqueeze(0).to(device)
    r = torch.tensor(qt_10k[rand_id], dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        eq = F.normalize(model_10k(q), dim=1)
        ep = F.normalize(model_10k(p), dim=1)
        er = F.normalize(model_10k(r), dim=1)
    gt_sims.append(F.cosine_similarity(eq, ep).item())
    rand_sims.append(F.cosine_similarity(eq, er).item())
print(f"Embedding quality — GT sim: {np.mean(gt_sims):.4f} | "
      f"Rand sim: {np.mean(rand_sims):.4f} | "
      f"Gap: {np.mean(gt_sims)-np.mean(rand_sims):.4f}")

# Build cosine index
idx_10k, build_10k, idxmem_10k = build_cosine_index(corpus_10k)
print(f"Index built: {build_10k:.1f}s | Mem: {idxmem_10k:.1f} MB")

results_10k = {}

# 1. No rerank — K=500
nbrs_500, qps = query_index(idx_10k, query_10k, k=500)
rec = eval_recall_all(gt_10k, nbrs_500, QUERY_START_10K)
results_10k['cosine_no_rerank'] = {**rec, 'qps': qps, 'build_s': build_10k,
                                    'vec_mb': vec_mb_10k, 'idx_mb': idxmem_10k}
print_results("Cosine (no rerank)", rec, qps, build_10k, vec_mb_10k, idxmem_10k, 512)

# 2. K=100 + WJ rerank
nbrs_100, qps_h = query_index(idx_10k, query_10k, k=100)
t0 = time.time()
nbrs_rr = rerank_wj(query_qt_10k, nbrs_100, corpus_qt_10k)
t_rr    = time.time() - t0
qps_rr  = len(query_10k) / (qps_h * len(query_10k) / 1000 / 1000 + t_rr)
# Simpler: total time
t0 = time.time()
nbrs_100_2, _ = query_index(idx_10k, query_10k, k=100)
t_hnsw = time.time() - t0
t0 = time.time()
nbrs_rr100 = rerank_wj(query_qt_10k, nbrs_100_2, corpus_qt_10k)
t_rr   = time.time() - t0
qps_rr100 = len(query_10k) / (t_hnsw + t_rr)
rec_rr100 = eval_recall_all(gt_10k, nbrs_rr100, QUERY_START_10K, max_k=100)
results_10k['cosine_k100_rerank'] = {**rec_rr100, 'qps': qps_rr100,
                                      'build_s': build_10k,
                                      'vec_mb': vec_mb_10k, 'idx_mb': idxmem_10k}
print_results("Cosine K=100 + WJ rerank", rec_rr100, qps_rr100,
              build_10k, vec_mb_10k, idxmem_10k, 512)

# 3. K=200 + WJ rerank
t0 = time.time()
nbrs_200, _ = query_index(idx_10k, query_10k, k=200)
t_hnsw = time.time() - t0
t0 = time.time()
nbrs_rr200 = rerank_wj(query_qt_10k, nbrs_200, corpus_qt_10k)
t_rr   = time.time() - t0
qps_rr200 = len(query_10k) / (t_hnsw + t_rr)
rec_rr200 = eval_recall_all(gt_10k, nbrs_rr200, QUERY_START_10K, max_k=200)
results_10k['cosine_k200_rerank'] = {**rec_rr200, 'qps': qps_rr200,
                                      'build_s': build_10k,
                                      'vec_mb': vec_mb_10k, 'idx_mb': idxmem_10k}
print_results("Cosine K=200 + WJ rerank", rec_rr200, qps_rr200,
              build_10k, vec_mb_10k, idxmem_10k, 512)

# 4. Sqrt/Bhattacharyya index
print("\n  Building sqrt/Bhattacharyya index...")
# Use WJ embeddings (ReLU+L1) then sqrt then L2-normalize
embs_wj_10k = []
with torch.no_grad():
    for start in range(0, len(qt_10k), 512):
        batch = torch.tensor(qt_10k[start:start+512],
                             dtype=torch.float32).to(device)
        emb   = model_10k(batch, for_index=True)  # ReLU + L1
        embs_wj_10k.append(emb.cpu().numpy())
embs_wj_10k = np.vstack(embs_wj_10k)

embs_sqrt_10k   = np.sqrt(embs_wj_10k)
norms           = np.linalg.norm(embs_sqrt_10k, axis=1, keepdims=True)
embs_sqrt_10k   = embs_sqrt_10k / np.maximum(norms, 1e-10)
corpus_sqrt_10k = embs_sqrt_10k[:QUERY_START_10K]
query_sqrt_10k  = embs_sqrt_10k[QUERY_START_10K:]

idx_sqrt_10k, build_sqrt, idxmem_sqrt = build_cosine_index(corpus_sqrt_10k)
nbrs_sqrt, qps_sqrt = query_index(idx_sqrt_10k, query_sqrt_10k, k=500)
rec_sqrt = eval_recall_all(gt_10k, nbrs_sqrt, QUERY_START_10K)
results_10k['bhattacharyya'] = {**rec_sqrt, 'qps': qps_sqrt,
                                 'build_s': build_sqrt, 'vec_mb': vec_mb_10k,
                                 'idx_mb': idxmem_sqrt}
print_results("Sqrt/Bhattacharyya (no rerank)", rec_sqrt, qps_sqrt,
              build_sqrt, vec_mb_10k, idxmem_sqrt, 512)

# ─── Full dataset experiment ───────────────────────────────────────────────────
print("\n" + "="*60)
print("MLP + COSINE — Full 233k dataset")
print("="*60)

model_full = QuadtreeCompressorV1Fixed(in_dim=qt_full.shape[1], out_dim=512).to(device)
model_full.load_state_dict(
    torch.load('/tmp/best_compressor_full_fixed.pt', weights_only=True))
model_full.eval()
print(f"Model loaded: {sum(p.numel() for p in model_full.parameters()):,} params")

embs_full    = generate_cosine_embeddings(model_full, qt_full, device)
corpus_full  = embs_full[:QUERY_START_ID]
query_full   = embs_full[QUERY_START_ID:]
vec_mb_full  = corpus_full.nbytes / 1024**2
print(f"Embeddings: {embs_full.shape} | Vec size: {vec_mb_full:.1f} MB")

# Embedding quality
gt_sims2, rand_sims2 = [], []
for i in range(200):
    qid    = QUERY_START_ID + i
    pos_id = gt_full.get(qid, [None])[0]
    if pos_id is None: continue
    rand_id = np.random.randint(0, QUERY_START_ID)
    q = torch.tensor(qt_full[qid], dtype=torch.float32).unsqueeze(0).to(device)
    p = torch.tensor(qt_full[pos_id], dtype=torch.float32).unsqueeze(0).to(device)
    r = torch.tensor(qt_full[rand_id], dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        eq = F.normalize(model_full(q), dim=1)
        ep = F.normalize(model_full(p), dim=1)
        er = F.normalize(model_full(r), dim=1)
    gt_sims2.append(F.cosine_similarity(eq, ep).item())
    rand_sims2.append(F.cosine_similarity(eq, er).item())
print(f"Embedding quality — GT sim: {np.mean(gt_sims2):.4f} | "
      f"Rand sim: {np.mean(rand_sims2):.4f} | "
      f"Gap: {np.mean(gt_sims2)-np.mean(rand_sims2):.4f}")

idx_full, build_full, idxmem_full = build_cosine_index(corpus_full)
print(f"Index built: {build_full:.1f}s | Mem: {idxmem_full:.1f} MB")

results_full = {}

# 1. No rerank K=500
nbrs_f500, qps_f = query_index(idx_full, query_full, k=500)
rec_f = eval_recall_all(gt_full, nbrs_f500, QUERY_START_ID)
results_full['cosine_no_rerank'] = {**rec_f, 'qps': qps_f,
                                     'build_s': build_full,
                                     'vec_mb': vec_mb_full, 'idx_mb': idxmem_full}
print_results("Cosine (no rerank)", rec_f, qps_f,
              build_full, vec_mb_full, idxmem_full, 512)

# 2. K=100 + WJ rerank
t0 = time.time()
nbrs_f100, _ = query_index(idx_full, query_full, k=100)
t_hnsw = time.time() - t0
t0 = time.time()
nbrs_rr_f100 = rerank_wj(query_qt_full, nbrs_f100, corpus_qt_full)
t_rr = time.time() - t0
qps_rrf100 = len(query_full) / (t_hnsw + t_rr)
rec_rrf100  = eval_recall_all(gt_full, nbrs_rr_f100, QUERY_START_ID, max_k=100)
results_full['cosine_k100_rerank'] = {**rec_rrf100, 'qps': qps_rrf100,
                                       'build_s': build_full,
                                       'vec_mb': vec_mb_full, 'idx_mb': idxmem_full}
print_results("Cosine K=100 + WJ rerank", rec_rrf100, qps_rrf100,
              build_full, vec_mb_full, idxmem_full, 512)

# 3. K=200 + WJ rerank
t0 = time.time()
nbrs_f200, _ = query_index(idx_full, query_full, k=200)
t_hnsw = time.time() - t0
t0 = time.time()
nbrs_rr_f200 = rerank_wj(query_qt_full, nbrs_f200, corpus_qt_full)
t_rr = time.time() - t0
qps_rrf200 = len(query_full) / (t_hnsw + t_rr)
rec_rrf200  = eval_recall_all(gt_full, nbrs_rr_f200, QUERY_START_ID, max_k=200)
results_full['cosine_k200_rerank'] = {**rec_rrf200, 'qps': qps_rrf200,
                                       'build_s': build_full,
                                       'vec_mb': vec_mb_full, 'idx_mb': idxmem_full}
print_results("Cosine K=200 + WJ rerank", rec_rrf200, qps_rrf200,
              build_full, vec_mb_full, idxmem_full, 512)

# ─── Final comparison table ───────────────────────────────────────────────────
print(f"\n{'='*100}")
print(f"MLP + COSINE FINAL SUMMARY ({THREADS} threads)")
print(f"{'='*100}")
print(f"{'Method':<35} {'R@10':>7} {'R@50':>7} {'R@100':>7} {'R@500':>7} "
      f"{'QPS':>8} {'Build':>8} {'Vec(MB)':>9} {'Idx(MB)':>9}")
print("-"*100)

def print_row(label, res, k_list):
    r10  = f"{res.get('R@10',  float('nan')):>7.4f}" if 10  in k_list else f"{'—':>7}"
    r50  = f"{res.get('R@50',  float('nan')):>7.4f}" if 50  in k_list else f"{'—':>7}"
    r100 = f"{res.get('R@100', float('nan')):>7.4f}" if 100 in k_list else f"{'—':>7}"
    r500 = f"{res.get('R@500', float('nan')):>7.4f}" if 500 in k_list else f"{'—':>7}"
    print(f"{label:<35} {r10} {r50} {r100} {r500} "
          f"{res['qps']:>8.1f} {res['build_s']:>7.1f}s "
          f"{res['vec_mb']:>8.1f} {res['idx_mb']:>8.1f}")

print("--- 10k dataset ---")
print_row("Cosine (no rerank)",        results_10k['cosine_no_rerank'],  [10,50,100,500])
print_row("Cosine K=100+WJrerank",     results_10k['cosine_k100_rerank'],[10,50,100])
print_row("Cosine K=200+WJrerank",     results_10k['cosine_k200_rerank'],[10,50,100])
print_row("Sqrt/Bhattacharyya",        results_10k['bhattacharyya'],     [10,50,100,500])
print("--- Full 233k dataset ---")
print_row("Cosine (no rerank)",        results_full['cosine_no_rerank'],  [10,50,100,500])
print_row("Cosine K=100+WJrerank",     results_full['cosine_k100_rerank'],[10,50,100])
print_row("Cosine K=200+WJrerank",     results_full['cosine_k200_rerank'],[10,50,100])

# Save results
with open('/tmp/results_mlp_cosine.pkl', 'wb') as f:
    pickle.dump({'10k': results_10k, 'full': results_full}, f)
print(f"\nResults saved to /tmp/results_mlp_cosine.pkl")

Loading cached data...
10k:  corpus=(8000, 18499) | queries=(2000, 18499)
Full: corpus=(187019, 18220) | queries=(46754, 18220)

MLP + COSINE — 10k dataset
Model loaded: 80,501,760 params


Embedding: 100%|██████████| 20/20 [00:00<00:00, 35.93it/s]


Embeddings: (10000, 512) | Vec size: 15.6 MB
Embedding quality — GT sim: 0.9914 | Rand sim: 0.7647 | Gap: 0.2268



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

Index built: 0.2s | Mem: 44.8 MB

  Cosine (no rerank)
    Dim=512 | Vec=15.6MB | Idx=44.8MB | Build=0.2s | QPS=30149.6
    R@10   = 0.6655
    R@50   = 0.8080
    R@100  = 0.8464
    R@500  = 0.9540

  Cosine K=100 + WJ rerank
    Dim=512 | Vec=15.6MB | Idx=44.8MB | Build=0.2s | QPS=405.8
    R@10   = 0.9946
    R@50   = 0.9581
    R@100  = 0.8464

  Cosine K=200 + WJ rerank
    Dim=512 | Vec=15.6MB | Idx=44.8MB | Build=0.2s | QPS=160.6
    R@10   = 0.9962
    R@50   = 0.9933
    R@100  = 0.9720

  Building sqrt/Bhattacharyya index...


...


  Sqrt/Bhattacharyya (no rerank)
    Dim=512 | Vec=15.6MB | Idx=19.5MB | Build=0.2s | QPS=28225.7
    R@10   = 0.6256
    R@50   = 0.7741
    R@100  = 0.8201
    R@500  = 0.9461

MLP + COSINE — Full 233k dataset
Model loaded: 79,358,976 params


Embedding: 100%|██████████| 457/457 [00:08<00:00, 54.41it/s]


Embeddings: (233773, 512) | Vec size: 365.3 MB
Embedding quality — GT sim: 0.9338 | Rand sim: 0.1255 | Gap: 0.8083



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

Index built: 123.5s | Mem: 577.6 MB

  Cosine (no rerank)
    Dim=512 | Vec=365.3MB | Idx=577.6MB | Build=123.5s | QPS=2289.3
    R@10   = 0.6581
    R@50   = 0.7264
    R@100  = 0.7398
    R@500  = 0.7643

  Cosine K=100 + WJ rerank
    Dim=512 | Vec=365.3MB | Idx=577.6MB | Build=123.5s | QPS=336.4
    R@10   = 0.9875
    R@50   = 0.9152
    R@100  = 0.7398

  Cosine K=200 + WJ rerank
    Dim=512 | Vec=365.3MB | Idx=577.6MB | Build=123.5s | QPS=154.6
    R@10   = 0.9917
    R@50   = 0.9805
    R@100  = 0.9258

MLP + COSINE FINAL SUMMARY (32 threads)
Method                                 R@10    R@50   R@100   R@500      QPS    Build   Vec(MB)   Idx(MB)
----------------------------------------------------------------------------------------------------
--- 10k dataset ---
Cosine (no rerank)                      nan     nan     nan     nan  30149.6     0.2s     15.6     44.8
Cosine K=100+WJrerank                   nan     nan     nan       —    405.8     0.2s     15.6     44.8
Cosine K

In [2]:
# Load saved results and print corrected table
import pickle

with open('/tmp/results_mlp_cosine.pkl', 'rb') as f:
    saved = pickle.load(f)

r10k  = saved['10k']
rfull = saved['full']

print(f"\n{'='*105}")
print(f"MLP + COSINE FINAL SUMMARY (32 threads)")
print(f"{'='*105}")
print(f"{'Method':<35} {'R@10':>7} {'R@50':>7} {'R@100':>7} {'R@500':>7} "
      f"{'QPS':>8} {'Build':>7} {'Vec(MB)':>8} {'Idx(MB)':>8}")
print("-"*105)

def fmt(val, K, valid_ks):
    if K not in valid_ks: return f"{'—':>7}"
    return f"{val:>7.4f}" if val == val else f"{'—':>7}"  # nan check

rows_10k = [
    ("10k | Cosine (no rerank)",    r10k['cosine_no_rerank'],   [10,50,100,500]),
    ("10k | K=100+WJrerank",        r10k['cosine_k100_rerank'], [10,50,100]),
    ("10k | K=200+WJrerank",        r10k['cosine_k200_rerank'], [10,50,100]),
    ("10k | Sqrt/Bhattacharyya",    r10k['bhattacharyya'],      [10,50,100,500]),
]
rows_full = [
    ("Full | Cosine (no rerank)",   rfull['cosine_no_rerank'],   [10,50,100,500]),
    ("Full | K=100+WJrerank",       rfull['cosine_k100_rerank'], [10,50,100]),
    ("Full | K=200+WJrerank",       rfull['cosine_k200_rerank'], [10,50,100]),
]

for label, res, valid_k in rows_10k + rows_full:
    r10  = res.get(10,  res.get('R@10',  float('nan')))
    r50  = res.get(50,  res.get('R@50',  float('nan')))
    r100 = res.get(100, res.get('R@100', float('nan')))
    r500 = res.get(500, res.get('R@500', float('nan')))
    qps  = res['qps']
    bld  = res['build_s']
    vec  = res['vec_mb']
    idx  = res['idx_mb']

    r10s  = f"{r10:.4f}"  if 10  in valid_k and r10==r10  else "—"
    r50s  = f"{r50:.4f}"  if 50  in valid_k and r50==r50  else "—"
    r100s = f"{r100:.4f}" if 100 in valid_k and r100==r100 else "—"
    r500s = f"{r500:.4f}" if 500 in valid_k and r500==r500 else "—"

    print(f"{label:<35} {r10s:>7} {r50s:>7} {r100s:>7} {r500s:>7} "
          f"{qps:>8.1f} {bld:>6.1f}s {vec:>8.1f} {idx:>8.1f}")


MLP + COSINE FINAL SUMMARY (32 threads)
Method                                 R@10    R@50   R@100   R@500      QPS   Build  Vec(MB)  Idx(MB)
---------------------------------------------------------------------------------------------------------
10k | Cosine (no rerank)             0.6655  0.8080  0.8464  0.9540  30149.6    0.2s     15.6     44.8
10k | K=100+WJrerank                 0.9946  0.9581  0.8464       —    405.8    0.2s     15.6     44.8
10k | K=200+WJrerank                 0.9962  0.9933  0.9720       —    160.6    0.2s     15.6     44.8
10k | Sqrt/Bhattacharyya             0.6256  0.7741  0.8201  0.9461  28225.7    0.2s     15.6     19.5
Full | Cosine (no rerank)            0.6581  0.7264  0.7398  0.7643   2289.3  123.5s    365.3    577.6
Full | K=100+WJrerank                0.9875  0.9152  0.7398       —    336.4  123.5s    365.3    577.6
Full | K=200+WJrerank                0.9917  0.9805  0.9258       —    154.6  123.5s    365.3    577.6
